# PROJENER — Notebook experimental integrado T1–T5

**Objetivo:** reunir prototipos experimentales para T1 (PPO didáctico), T2 (CNN-Transformer), T3 (VAE), T4 (cWGAN-GP) y T5 (CNN convencional de referencia).

**Importante para el TFM:** los modelos son prototipos de investigación, no sistemas operativos de guerra electrónica. T1 usa un entorno sintético de decisión, no controla equipos ni interferencias. T4 genera muestras sintéticas y no demuestra por sí mismo mejoras. T5 es una baseline convencional; **no está certificada DO-178C**. No presentes métricas hasta ejecutar el notebook y guardar las salidas reales.

## 1. Configuración y reproducibilidad

In [ ]:
import os, pickle, random, time, copy, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

SEED = 42
MAX_PER_CLASS = 500       # reducir para pruebas rápidas; None para usar todas las muestras
EPOCHS_CLF = 5            # aumentar para el experimento final
EPOCHS_VAE = 5
EPOCHS_GAN = 3
PPO_UPDATES = 100
BATCH_SIZE = 128
TEST_SIZE = 0.20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Dispositivo:", DEVICE, "| PyTorch:", torch.__version__)

## 2. Cargar y validar RadioML2016.10a

Coloca el archivo recuperado `RML2016.10a_dict_optimized.pkl` en `/content/radioml_recuperado/` o selecciona su ruta en `DATA_PATH`. El código comprueba estructura, valores finitos y forma. No usa el archivo truncado `RML2016.10a_dict-2.pkl` salvo que cambies la ruta expresamente.

In [ ]:
DATA_PATH = "/content/radioml_recuperado/RML2016.10a_dict_optimized.pkl"

def cargar_radioml(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"No se encuentra {path}. Sube el pickle íntegro o cambia DATA_PATH.")
    with open(path, "rb") as f:
        raw = pickle.load(f, encoding="latin1")
    Xs, ys, snr_list = [], [], []
    mods = sorted({str(k[0]) for k in raw.keys()})
    mod_to_id = {m:i for i,m in enumerate(mods)}
    for (mod, snr), arr in raw.items():
        a = np.asarray(arr, dtype=np.float32)
        if a.ndim != 3 or a.shape[1] != 2:
            raise ValueError(f"Forma inesperada para {(mod,snr)}: {a.shape}; se esperaba (N,2,L)")
        if not np.isfinite(a).all():
            raise ValueError(f"Hay NaN/Inf en {(mod,snr)}")
        if MAX_PER_CLASS is not None: a = a[:MAX_PER_CLASS]
        Xs.append(a); ys.extend([mod_to_id[str(mod)]]*len(a)); snr_list.extend([int(snr)]*len(a))
    X = np.concatenate(Xs).astype(np.float32)
    y = np.asarray(ys, dtype=np.int64); snr = np.asarray(snr_list, dtype=np.int64)
    # Normalización por muestra, evitando dividir por cero
    escala = np.sqrt(np.mean(X**2, axis=(1,2), keepdims=True) + 1e-8)
    X = X / escala
    print("Dataset validado:", X.shape, "| clases:", len(mods), "| SNR:", sorted(np.unique(snr).tolist()))
    print("Valores finitos:", bool(np.isfinite(X).all()), "| clases:", mods)
    return X, y, snr, mods

X, y, snr, MODULACIONES = cargar_radioml(DATA_PATH)
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
snr_train, snr_test = snr[idx_train], snr[idx_test]
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=BATCH_SIZE, shuffle=False)
N_CLASSES = len(MODULACIONES); SIGNAL_LEN = X.shape[-1]
print("Train:", len(idx_train), "Test:", len(idx_test), "Longitud señal:", SIGNAL_LEN)

## 3. Definición de T2 y T5: clasificador CNN-Transformer y baseline CNN

In [ ]:
class CNNBaseline(nn.Module):
    """T5: baseline CNN convencional; NO es una implementación certificada DO-178C."""
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2,32,7,padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32,64,5,padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64,128,3,padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(128,n_classes)
    def forward(self,x): return self.fc(self.features(x).squeeze(-1))

class CNNTransformer(nn.Module):
    """T2: convoluciones 1D + TransformerEncoder para clasificación de modulaciones."""
    def __init__(self,n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(2,64,5,padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
                                  nn.Conv1d(64,64,3,padding=1), nn.BatchNorm1d(64), nn.ReLU())
        layer = nn.TransformerEncoderLayer(d_model=64,nhead=4,dim_feedforward=128,dropout=.1,batch_first=True)
        self.transformer = nn.TransformerEncoder(layer,num_layers=2)
        self.fc = nn.Sequential(nn.LayerNorm(64),nn.Linear(64,n_classes))
    def forward(self,x):
        z=self.conv(x).transpose(1,2); z=self.transformer(z)
        return self.fc(z.mean(dim=1))

for nombre, modelo in [("T2 CNN-Transformer",CNNTransformer(N_CLASSES)),("T5 CNN baseline",CNNBaseline(N_CLASSES))]:
    print(nombre, "parámetros entrenables:", sum(p.numel() for p in modelo.parameters() if p.requires_grad))

## 4. Entrenamiento y evaluación común de T2 y T5

Se comparten particiones, métrica y procedimiento. Los resultados se generan al ejecutar esta celda; no están precargados ni inventados.

In [ ]:
def entrenar_clasificador(modelo, nombre, epochs=EPOCHS_CLF):
    modelo=modelo.to(DEVICE); opt=torch.optim.AdamW(modelo.parameters(),lr=1e-3,weight_decay=1e-4)
    crit=nn.CrossEntropyLoss(); historial=[]; mejor=0.0; mejor_estado=None
    for ep in range(1,epochs+1):
        t0=time.perf_counter(); modelo.train(); suma=0; n=0
        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); logits=modelo(xb); loss=crit(logits,yb); loss.backward(); opt.step()
            suma+=loss.item()*len(yb); n+=len(yb)
        modelo.eval(); yt=[]; yp=[]; vloss=0; nv=0
        with torch.no_grad():
            for xb,yb in test_loader:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE); logits=modelo(xb); vloss+=crit(logits,yb).item()*len(yb); nv+=len(yb)
                yt.extend(yb.cpu().numpy()); yp.extend(logits.argmax(1).cpu().numpy())
        acc=accuracy_score(yt,yp); historial.append({"época":ep,"loss_train":suma/n,"loss_test":vloss/nv,"accuracy_test":acc,"segundos":time.perf_counter()-t0})
        if acc>mejor: mejor=acc; mejor_estado=copy.deepcopy(modelo.state_dict())
        print(f"{nombre} | época {ep}/{epochs} | loss train={suma/n:.4f} | loss test={vloss/nv:.4f} | accuracy={acc:.4f}")
    if mejor_estado is not None: modelo.load_state_dict(mejor_estado)
    modelo.eval(); yt=[]; yp=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            pred=modelo(xb.to(DEVICE)).argmax(1).cpu().numpy(); yp.extend(pred); yt.extend(yb.numpy())
    return modelo, pd.DataFrame(historial), np.array(yt), np.array(yp)

resultados_clf={}; modelos_clf={}
for nombre, modelo in [("T2 CNN-Transformer",CNNTransformer(N_CLASSES)),("T5 CNN baseline",CNNBaseline(N_CLASSES))]:
    m,h,yt,yp=entrenar_clasificador(modelo,nombre); modelos_clf[nombre]=m; resultados_clf[nombre]={"historial":h,"y_true":yt,"y_pred":yp,"accuracy":accuracy_score(yt,yp)}
print("\nResumen REAL de la ejecución:")
print(pd.DataFrame([{"sistema":k,"accuracy_test":v["accuracy"]} for k,v in resultados_clf.items()]))

In [ ]:
# Matrices de confusión y curvas de aprendizaje de T2 y T5
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax,(nombre,res) in zip(axes,resultados_clf.items()):
    ConfusionMatrixDisplay(confusion_matrix(res["y_true"],res["y_pred"]),display_labels=MODULACIONES).plot(ax=ax,xticks_rotation=90,colorbar=False)
    ax.set_title(f"{nombre} | acc={res['accuracy']:.3f}")
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
for nombre,res in resultados_clf.items(): plt.plot(res["historial"]["época"],res["historial"]["accuracy_test"],marker="o",label=nombre)
plt.xlabel("Época"); plt.ylabel("Accuracy en test"); plt.grid(True); plt.legend(); plt.title("Evolución de accuracy (test)"); plt.show()
for nombre,res in resultados_clf.items():
    print("\n",nombre,"\n",classification_report(res["y_true"],res["y_pred"],target_names=MODULACIONES,zero_division=0))

## 5. T3 — VAE para detección experimental de anomalías

El VAE aprende a reconstruir señales del conjunto de entrenamiento. El error de reconstrucción se utiliza como puntuación de anomalía. RadioML no etiqueta por defecto señales como legítimas o spoofing: por tanto, **no calculamos AUC de spoofing sin etiquetas de anomalía independientes**.

In [ ]:
class VAE(nn.Module):
    def __init__(self, signal_len, latent_dim=32):
        super().__init__(); d=2*signal_len
        self.enc=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Linear(256,128),nn.ReLU())
        self.mu=nn.Linear(128,latent_dim); self.logvar=nn.Linear(128,latent_dim)
        self.dec=nn.Sequential(nn.Linear(latent_dim,128),nn.ReLU(),nn.Linear(128,256),nn.ReLU(),nn.Linear(256,d))
    def forward(self,x):
        b=x.shape[0]; h=self.enc(x.reshape(b,-1)); mu=self.mu(h); logvar=self.logvar(h)
        z=mu+torch.randn_like(mu)*torch.exp(.5*logvar) if self.training else mu
        return self.dec(z).reshape_as(x),mu,logvar

vae=VAE(SIGNAL_LEN).to(DEVICE); opt_vae=torch.optim.Adam(vae.parameters(),lr=1e-3); hist_vae=[]
for ep in range(1,EPOCHS_VAE+1):
    vae.train(); total=0; count=0
    for xb,_ in train_loader:
        xb=xb.to(DEVICE); opt_vae.zero_grad(); recon,mu,logvar=vae(xb)
        rec=F.mse_loss(recon,xb,reduction="mean"); kl=-.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp()); loss=rec+1e-3*kl
        loss.backward(); opt_vae.step(); total+=loss.item()*len(xb); count+=len(xb)
    hist_vae.append(total/count); print(f"T3 VAE época {ep}/{EPOCHS_VAE} | loss={total/count:.6f}")

vae.eval(); scores=[]
with torch.no_grad():
    for xb,_ in test_loader:
        xb=xb.to(DEVICE); recon,_,_=vae(xb); scores.extend(((recon-xb)**2).mean(dim=(1,2)).cpu().numpy())
vae_scores=np.asarray(scores)
print("Error de reconstrucción en test — resumen descriptivo:")
print(pd.Series(vae_scores).describe())
plt.figure(figsize=(7,4)); plt.hist(vae_scores,bins=40); plt.xlabel("Error MSE de reconstrucción"); plt.ylabel("Número de señales"); plt.title("T3 VAE: distribución del error (test)"); plt.show()

## 6. T4 — cWGAN-GP condicional para generación de señales sintéticas

Prototipo compacto condicionado por clase. Entrenarlo puede consumir GPU y tiempo; el valor predeterminado es pocas épocas. La generación sintética no debe mezclarse con test ni presentarse como mejora hasta evaluar su efecto en un experimento separado.

In [ ]:
LATENT_DIM=64
class Generator(nn.Module):
    def __init__(self,n_classes,signal_len):
        super().__init__(); self.emb=nn.Embedding(n_classes,16); self.net=nn.Sequential(nn.Linear(LATENT_DIM+16,256),nn.ReLU(),nn.Linear(256,512),nn.ReLU(),nn.Linear(512,2*signal_len),nn.Tanh()); self.signal_len=signal_len
    def forward(self,z,label): return self.net(torch.cat([z,self.emb(label)],dim=1)).reshape(-1,2,self.signal_len)
class Critic(nn.Module):
    def __init__(self,n_classes,signal_len):
        super().__init__(); self.emb=nn.Embedding(n_classes,16); self.net=nn.Sequential(nn.Linear(2*signal_len+16,512),nn.LeakyReLU(.2),nn.Linear(512,256),nn.LeakyReLU(.2),nn.Linear(256,1)); self.signal_len=signal_len
    def forward(self,x,label): return self.net(torch.cat([x.reshape(x.shape[0],-1),self.emb(label)],dim=1)).squeeze(1)

def gradient_penalty(critic,real,fake,labels):
    a=torch.rand(real.size(0),1,1,device=DEVICE); mixed=(a*real+(1-a)*fake).requires_grad_(True)
    out=critic(mixed,labels); grad=torch.autograd.grad(out,mixed,torch.ones_like(out),create_graph=True,retain_graph=True)[0]
    return ((grad.reshape(grad.size(0),-1).norm(2,dim=1)-1)**2).mean()

G=Generator(N_CLASSES,SIGNAL_LEN).to(DEVICE); C=Critic(N_CLASSES,SIGNAL_LEN).to(DEVICE)
optG=torch.optim.Adam(G.parameters(),lr=1e-4,betas=(.0,.9)); optC=torch.optim.Adam(C.parameters(),lr=1e-4,betas=(.0,.9)); GP_WEIGHT=10; N_CRITIC=3
for ep in range(1,EPOCHS_GAN+1):
    c_losses=[]; g_losses=[]
    for step,(real,labels) in enumerate(train_loader):
        real,labels=real.to(DEVICE),labels.to(DEVICE); bs=len(real)
        for _ in range(N_CRITIC):
            z=torch.randn(bs,LATENT_DIM,device=DEVICE); fake=G(z,labels).detach()
            lossC=C(fake,labels).mean()-C(real,labels).mean()+GP_WEIGHT*gradient_penalty(C,real,fake,labels)
            optC.zero_grad(); lossC.backward(); optC.step(); c_losses.append(lossC.item())
        z=torch.randn(bs,LATENT_DIM,device=DEVICE); fake=G(z,labels); lossG=-C(fake,labels).mean()
        optG.zero_grad(); lossG.backward(); optG.step(); g_losses.append(lossG.item())
    print(f"T4 cWGAN-GP época {ep}/{EPOCHS_GAN} | loss crítico={np.mean(c_losses):.4f} | loss generador={np.mean(g_losses):.4f}")

@torch.no_grad()
def generar_sinteticas(clase, n=8):
    G.eval(); z=torch.randn(n,LATENT_DIM,device=DEVICE); labels=torch.full((n,),int(clase),device=DEVICE,dtype=torch.long)
    return G(z,labels).cpu().numpy()

# Visualiza algunas muestras sintéticas de la primera clase; esto NO es una métrica de calidad.
sinteticas=generar_sinteticas(0,4)
fig,axs=plt.subplots(2,2,figsize=(10,5))
for i,ax in enumerate(axs.flat): ax.plot(sinteticas[i,0],label="I"); ax.plot(sinteticas[i,1],label="Q"); ax.set_title(f"T4 muestra sintética {i+1}"); ax.legend()
plt.tight_layout(); plt.show()

## 7. T1 — PPO en un entorno sintético de decisión

Este módulo es un ejemplo didáctico para demostrar el bucle PPO (política, recompensa, ventaja y objetivo recortado). Las acciones son decisiones abstractas de clasificación: **aceptar predicción**, **solicitar revisión** o **marcar incertidumbre**. No actúa sobre transmisores, radares, interferencias ni equipos reales. La recompensa se genera sintéticamente y no es comparable con accuracy de T2/T5.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self,state_dim=3,n_actions=3):
        super().__init__(); self.body=nn.Sequential(nn.Linear(state_dim,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh()); self.actor=nn.Linear(64,n_actions); self.critic=nn.Linear(64,1)
    def forward(self,s):
        h=self.body(s); return torch.distributions.Categorical(logits=self.actor(h)),self.critic(h).squeeze(-1)

def entorno_sintetico(n=256):
    # Estado: confianza del clasificador, SNR normalizado, indicador de desacuerdo simulado.
    confianza=np.random.uniform(.35,1.0,n).astype(np.float32)
    snr_norm=np.random.uniform(0,1,n).astype(np.float32)
    desacuerdo=np.random.binomial(1,.2,n).astype(np.float32)
    estado=np.stack([confianza,snr_norm,desacuerdo],axis=1)
    correcto=np.random.binomial(1,np.clip(.45+.5*confianza-.15*desacuerdo,0,1),n)
    # Acción 0 aceptar; 1 revisión; 2 marcar incertidumbre. Recompensas solo sintéticas.
    rewards=np.stack([np.where(correcto==1,1.0,-1.0),np.full(n,.15),np.where(confianza<.6,.25,-.1)],axis=1).astype(np.float32)
    return estado,rewards

policy=ActorCritic().to(DEVICE); optP=torch.optim.Adam(policy.parameters(),lr=3e-4); clip_eps=.2; gamma=.99; lam=.95
ppo_history=[]
for update in range(1,PPO_UPDATES+1):
    states_np,reward_table=entorno_sintetico(256); states=torch.tensor(states_np,device=DEVICE); rewards_t=torch.tensor(reward_table,device=DEVICE)
    with torch.no_grad():
        dist,values=policy(states); actions=dist.sample(); old_logp=dist.log_prob(actions); rewards=rewards_t[torch.arange(len(actions),device=DEVICE),actions]
        # Retorno de un paso y ventaja normalizada para el entorno de un paso
        returns=rewards; advantages=returns-values; advantages=(advantages-advantages.mean())/(advantages.std()+1e-8)
    for _ in range(4):
        dist,new_values=policy(states); new_logp=dist.log_prob(actions); ratio=torch.exp(new_logp-old_logp)
        obj1=ratio*advantages; obj2=torch.clamp(ratio,1-clip_eps,1+clip_eps)*advantages
        loss=-(torch.minimum(obj1,obj2)).mean()+.5*F.mse_loss(new_values,returns)-.01*dist.entropy().mean()
        optP.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(policy.parameters(),.5); optP.step()
    ppo_history.append({"update":update,"reward_media":float(rewards.mean().cpu()),"loss":float(loss.detach().cpu())})
    if update%20==0: print(f"T1 PPO update {update}/{PPO_UPDATES} | recompensa media={np.mean([x['reward_media'] for x in ppo_history[-20:]]):.3f}")

ppo_df=pd.DataFrame(ppo_history); plt.figure(figsize=(7,4)); plt.plot(ppo_df.update,ppo_df.reward_media); plt.xlabel("Actualización PPO"); plt.ylabel("Recompensa media por lote"); plt.title("T1 PPO — entorno sintético didáctico"); plt.grid(True); plt.show()

## 8. Resumen de salidas y guardado de resultados

Las métricas de T2/T5 son clasificación medida en el conjunto de test. T3 entrega errores de reconstrucción; T4 entrega pérdidas de entrenamiento y señales generadas; T1 entrega recompensas del entorno sintético. No agregamos estas métricas heterogéneas en una única puntuación.

In [ ]:
resumen = pd.DataFrame([{"sistema":k,"métrica":"accuracy_test","valor":v["accuracy"],"interpretación":"Clasificación en test"} for k,v in resultados_clf.items()])
resumen = pd.concat([resumen,pd.DataFrame([
    {"sistema":"T3 VAE","métrica":"MSE reconstrucción media test","valor":float(vae_scores.mean()),"interpretación":"Puntuación no supervisada; no es AUC"},
    {"sistema":"T1 PPO","métrica":"Recompensa media últimas 20 actualizaciones","valor":float(ppo_df.reward_media.tail(20).mean()),"interpretación":"Entorno sintético; no comparable con accuracy"}
])],ignore_index=True)
print(resumen.to_string(index=False))
OUT_DIR=Path("/content/projener_resultados") if Path("/content").exists() else Path("projener_resultados")
OUT_DIR.mkdir(parents=True,exist_ok=True)
resumen.to_csv(OUT_DIR/"resumen_metricas.csv",index=False)
for nombre,res in resultados_clf.items(): res["historial"].to_csv(OUT_DIR/(nombre.lower().replace(" ","_").replace("-","_")+"_historial.csv"),index=False)
ppo_df.to_csv(OUT_DIR/"t1_ppo_historial.csv",index=False)
pd.DataFrame({"error_reconstruccion":vae_scores}).to_csv(OUT_DIR/"t3_vae_scores_test.csv",index=False)
torch.save(modelos_clf["T2 CNN-Transformer"].state_dict(),OUT_DIR/"t2_cnn_transformer.pt")
torch.save(modelos_clf["T5 CNN baseline"].state_dict(),OUT_DIR/"t5_cnn_baseline_no_certificada.pt")
torch.save(vae.state_dict(),OUT_DIR/"t3_vae.pt"); torch.save(G.state_dict(),OUT_DIR/"t4_generator.pt"); torch.save(policy.state_dict(),OUT_DIR/"t1_ppo_actor_critic.pt")
print("Resultados guardados en:",OUT_DIR.resolve())

## 9. Qué puedes afirmar en la memoria

- **T1:** se ha desarrollado un prototipo PPO en entorno sintético; no valida decisiones operacionales reales.
- **T2:** clasificador CNN-Transformer entrenado y evaluado con RadioML; reporta las métricas obtenidas al ejecutar.
- **T3:** VAE entrenado para reconstrucción; sin etiquetas de anomalía independientes no se afirma una tasa de detección ni AUC de spoofing.
- **T4:** cWGAN-GP condicional entrenada como prototipo generativo; antes de usar datos sintéticos para aumentar el entrenamiento, evaluar diversidad, fidelidad y efecto en test real.
- **T5:** CNN convencional de referencia. No se debe denominar “certificada” sin evidencia documental de certificación y proceso de verificación correspondiente.

**Limitación de comparación:** no es metodológicamente correcto comparar directamente la recompensa PPO, el MSE del VAE, las pérdidas GAN y la accuracy de clasificación. Cada sistema requiere métricas específicas.